# AI Tutor Agent: RAG + LangGraph Prototype

This notebook builds a small interview-ready **AI Tutor Agent**.

The workflow is:

**Student Question → Retrieve Textbook Content → Generate Answer → Evaluate Groundedness → Retry if Needed**

It demonstrates:

- Text chunking
- Embeddings / vector search
- RAG
- LLM-style answer generation
- Groundedness evaluation
- LangGraph state
- Conditional routing
- Retry loops

The notebook is designed to run **without an API key** using a lightweight local fallback.
If you later want to use an OpenAI model, you can replace the `call_llm()` function.

## 1. Install dependencies

Run the cell below once.

If you are running this in an environment where packages are already installed, you can skip it.

In [ ]:
%pip install -q langgraph scikit-learn

## 2. Imports

In [ ]:
from typing import TypedDict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from langgraph.graph import StateGraph, START, END

## 3. Create a tiny textbook knowledge base

In a production system, these would come from PDFs, textbooks, course materials, or a content database.

For this prototype, we use a few short biology passages.

In [ ]:
documents = [
    {
        "id": "bio_1",
        "text": (
            "Photosynthesis is the process by which plants convert light energy "
            "into chemical energy. It primarily occurs in chloroplasts."
        )
    },
    {
        "id": "bio_2",
        "text": (
            "Chlorophyll is a pigment found in chloroplasts. It absorbs light, "
            "especially red and blue wavelengths, and plays an important role "
            "in photosynthesis."
        )
    },
    {
        "id": "bio_3",
        "text": (
            "During photosynthesis, plants use carbon dioxide and water to "
            "produce glucose and oxygen."
        )
    },
    {
        "id": "bio_4",
        "text": (
            "Glucose produced during photosynthesis stores chemical energy that "
            "plants can use for growth, metabolism, and other biological processes."
        )
    },
    {
        "id": "bio_5",
        "text": (
            "Cellular respiration is a process in which cells break down glucose "
            "to release energy. It is different from photosynthesis."
        )
    }
]

for d in documents:
    print(d["id"], ":", d["text"])

## 4. Build a lightweight retriever

A production RAG system often uses an embedding model and a vector database such as:

- FAISS
- Chroma
- Pinecone
- pgvector

To make this notebook easy to run, we use **TF-IDF vectors + cosine similarity**.

The architecture is the same:

**Question → Vector Representation → Similarity Search → Top-K Relevant Chunks**

In [ ]:
texts = [d["text"] for d in documents]

vectorizer = TfidfVectorizer(stop_words="english")
document_matrix = vectorizer.fit_transform(texts)

def retrieve_documents(question: str, top_k: int = 2):
    question_vector = vectorizer.transform([question])
    similarities = cosine_similarity(question_vector, document_matrix)[0]

    ranked_indices = similarities.argsort()[::-1][:top_k]

    results = []
    for idx in ranked_indices:
        results.append({
            "id": documents[idx]["id"],
            "text": documents[idx]["text"],
            "score": float(similarities[idx])
        })

    return results

### Test the retriever

In [ ]:
question = "Why do plants need sunlight?"

results = retrieve_documents(question)

for r in results:
    print(f"{r['id']} | score={r['score']:.3f}")
    print(r["text"])
    print()

## 5. Create an LLM interface

For an interview prototype, it is useful to separate the LLM call from the rest of the system.

Below, `call_llm()` is a **local fallback** that generates a simple grounded answer from retrieved context.

Later, you can replace this one function with an OpenAI, Anthropic, Gemini, or local-model API call without changing the rest of the LangGraph workflow.

In [ ]:
def call_llm(question: str, context: str) -> str:
    """
    Lightweight local fallback.

    In production, replace this function with a real LLM call.
    """
    q = question.lower()

    if "sunlight" in q or "light" in q:
        return (
            "Plants need sunlight because photosynthesis converts light energy "
            "into chemical energy. Chlorophyll absorbs light, and the energy is "
            "used to help produce glucose."
        )

    if "photosynthesis" in q:
        return (
            "Photosynthesis is the process by which plants convert light energy "
            "into chemical energy, using carbon dioxide and water to produce "
            "glucose and oxygen."
        )

    return (
        "Based on the retrieved textbook context: " 
        + context.split(".")[0].strip() 
        + "."
    )

## 6. Create a groundedness evaluator

In a production LLM system, an evaluator can be:

- another LLM
- a rule-based evaluator
- an entailment model
- a human reviewer
- a combination of these

For this prototype, we use a lightweight lexical overlap heuristic so the notebook works without an API key.

The evaluator returns:

- `PASS`
- `FAIL`

In [ ]:
def evaluate_groundedness(answer: str, context: str, min_overlap: int = 3) -> str:
    answer_words = {
        w.strip(".,!?;:").lower()
        for w in answer.split()
        if len(w.strip(".,!?;:")) > 4
    }

    context_words = {
        w.strip(".,!?;:").lower()
        for w in context.split()
        if len(w.strip(".,!?;:")) > 4
    }

    overlap = answer_words.intersection(context_words)

    return "PASS" if len(overlap) >= min_overlap else "FAIL"

## 7. Define LangGraph state

The shared state acts like the agent's working memory.

Each node reads from the state and returns updates to it.

In [ ]:
class TutorState(TypedDict, total=False):
    question: str
    context: str
    answer: str
    evaluation: str
    attempts: int
    retrieved_docs: list

## 8. Define the agent nodes

We will create three nodes:

1. `retrieve`
2. `generate`
3. `evaluate`

In [ ]:
def retrieve_node(state: TutorState):
    question = state["question"]
    attempts = state.get("attempts", 0) + 1

    # Increase top_k on later attempts to simulate broader retrieval.
    top_k = min(1 + attempts, 4)

    retrieved_docs = retrieve_documents(question, top_k=top_k)
    context = "\n\n".join(d["text"] for d in retrieved_docs)

    print(f"[retrieve] attempt={attempts}, top_k={top_k}")

    return {
        "retrieved_docs": retrieved_docs,
        "context": context,
        "attempts": attempts
    }


def generate_node(state: TutorState):
    answer = call_llm(
        question=state["question"],
        context=state["context"]
    )

    print("[generate]", answer)

    return {
        "answer": answer
    }


def evaluate_node(state: TutorState):
    result = evaluate_groundedness(
        answer=state["answer"],
        context=state["context"]
    )

    print("[evaluate]", result)

    return {
        "evaluation": result
    }

## 9. Define conditional routing

If the answer is grounded, stop.

If it fails, retrieve more context and try again.

We also cap retries at 3 attempts so the graph cannot loop forever.

In [ ]:
def route_after_evaluation(state: TutorState):
    if state["evaluation"] == "PASS":
        return "finish"

    if state.get("attempts", 0) >= 3:
        return "finish"

    return "retry"

## 10. Build the LangGraph workflow

In [ ]:
graph = StateGraph(TutorState)

graph.add_node("retrieve", retrieve_node)
graph.add_node("generate", generate_node)
graph.add_node("evaluate", evaluate_node)

graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", "evaluate")

graph.add_conditional_edges(
    "evaluate",
    route_after_evaluation,
    {
        "finish": END,
        "retry": "retrieve"
    }
)

app = graph.compile()

print("LangGraph workflow compiled successfully.")

## 11. Run the AI Tutor Agent

In [ ]:
result = app.invoke({
    "question": "Why do plants need sunlight?",
    "attempts": 0
})

print("\nFINAL ANSWER:")
print(result["answer"])

print("\nEVALUATION:")
print(result["evaluation"])

print("\nATTEMPTS:")
print(result["attempts"])

print("\nRETRIEVED DOCUMENTS:")
for d in result["retrieved_docs"]:
    print("-", d["id"], f"(score={d['score']:.3f})")

## 12. Try another question

In [ ]:
result = app.invoke({
    "question": "What does photosynthesis produce?",
    "attempts": 0
})

print("\nFINAL ANSWER:")
print(result["answer"])

# How to explain this project in an interview

You can say:

> I built a small AI tutoring agent using a RAG architecture and LangGraph.  
> When a student asks a question, the system first retrieves relevant educational content using vector-based similarity search. The retrieved content is then passed to an LLM-style generation step to produce an answer.  
> I also added a groundedness evaluation node. If the answer is not sufficiently supported by the retrieved context, LangGraph conditionally routes the workflow back to retrieval and expands the context before generating again.  
> This project helped me work with RAG, state management, conditional routing, retry loops, and LLM evaluation in an agentic workflow.

## Architecture

```text
Student Question
       |
       v
   Retriever
       |
       v
Relevant Context
       |
       v
    Generator
       |
       v
     Answer
       |
       v
   Evaluator
    /     \
 PASS     FAIL
  |         |
  v         |
 END <------+
      retry retrieval
```

# How to make this more production-like

For a real project, the next upgrades would be:

1. Replace TF-IDF with a real embedding model.
2. Store embeddings in FAISS, Chroma, Pinecone, or pgvector.
3. Replace `call_llm()` with a real LLM API.
4. Use structured output for the evaluator.
5. Add source citations to every answer.
6. Log latency, token usage, retrieval scores, and evaluation results.
7. Add prompt-injection safeguards.
8. Build an offline evaluation dataset with known questions and answers.
9. Track retrieval metrics such as Recall@K and generation metrics such as groundedness.
10. Add human review for low-confidence or high-risk answers.